In [1]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Literal
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel , Field

In [2]:
load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b')

In [3]:
class SenitmentSchema(BaseModel):

    sentiment: Literal['positive' , 'negative'] = Field(description='sentiment of the review')

In [4]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal['UX' , 'Performance' , 'Supoort' , 'Bug' , 'Other'] = Field(description="The category of the issue mentioned in the review")
    tone: Literal['angry' , 'frustrated' , 'disappointed' , 'calm'] = Field(description="The emotional tone expressed by the user")
    urgency: Literal['Low' , 'Medium' , 'High'] = Field(description="How urgent or critical the issue appears to be")

In [5]:
structured_model = model.with_structured_output(SenitmentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [6]:
prompt = 'what is the sentiment of this review - The software is bad'
structured_model.invoke(prompt)

SenitmentSchema(sentiment='negative')

In [7]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal['positive' , 'negative']
    diagnose : dict
    response : str

In [8]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review analyze its sentiment \n {state['review']}'
    sentiment = structured_model.invoke(prompt).sentiment

    return{'sentiment' : sentiment}


In [9]:
def positive_response(state: ReviewState):
    prompt = f""" 
            write a warm thank you meassage in response to this review \n\n{state['review']}\n
Also, kindly ask the user to leave the review on our website
"""

    response = model.invoke(prompt).content
    return {'response' : response}

In [10]:
def run_diagnosis(state: ReviewState):
    prompt = f""" 
                Diagnose this Negative review - \n\n{state['review']}\n
                Return issue type , tone and urgency"""

    response = structured_model2.invoke(prompt)
    return {'diagnose' : response.model_dump()}

In [11]:
def negative_response(state: ReviewState):

    diagnosis = state['diagnose']

    prompt = f""" 
                You are a support assistant
                the user had a {diagnosis['issue_type']} issue , sounded {diagnosis['tone']}, and marked iths urgency as {diagnosis['urgency']}
                Write an empathic , resolution message """

    response = model.invoke(prompt).content
    return {'response' : response}

In [12]:
def check_sentiment(state: ReviewState) -> Literal['positive_response' , 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [13]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment' , find_sentiment)
graph.add_node('positive_response' , positive_response)
graph.add_node('run_diagnosis' , run_diagnosis)
graph.add_node('negative_response' , negative_response)


graph.add_edge(START , 'find_sentiment')
graph.add_conditional_edges('find_sentiment' , check_sentiment)

graph.add_edge('positive_response' , END)
graph.add_edge('run_diagnosis' , 'negative_response')
graph.add_edge('negative_response' , END)

workflow = graph.compile()

In [14]:
initial_state = {
    'review': 'I’ve been using this app for about a month now, and I’m honestly disappointed. The app crashes every time I try to upload a file, and the search feature is painfully slow. I reached out to support twice and never got a reply. For something I’m paying for, this is unacceptable. Please fix these issues soon.'
}

workflow.invoke(initial_state)

{'review': 'I’ve been using this app for about a month now, and I’m honestly disappointed. The app crashes every time I try to upload a file, and the search feature is painfully slow. I reached out to support twice and never got a reply. For something I’m paying for, this is unacceptable. Please fix these issues soon.',
 'sentiment': 'negative',
 'diagnose': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'High'},
 'response': 'Hi there,\n\nI’m really sorry for the frustration you’ve experienced with this bug—especially when it’s urgent and impacting your workflow. I’ve taken a close look at the issue and have implemented a fix that should resolve the problem you were seeing.\n\n**What’s been fixed?**  \n- The underlying logic that was causing the error when you performed [specific action] has been corrected.  \n- We’ve also added additional validation to prevent this scenario from re‑occurring.\n\n**What you need to do next:**  \n1. **Refresh the page** (or restart the app if y